#### 1. Тема та мета роботи.
##### Тема:
Вибір та обґрунтування алгоритмічної стратегії розроблення програмного забезпечення (Варіант 18: Оптимізація пошуку у векторній базі даних та системах Retrieval-Augmented Generation (RAG)).
##### Мета:
Набути практичних навичок аналізу алгоритмічних стратегій розв'язання задач, визначення умов їх ефективного застосування та обґрунтованого вибору алгоритмічної стратегії залежно від функціональних і нефункціональних вимог програмної системи.

#### 2. Постановка завдання.
Для обраної задачі (Варіант 18. Оптимізація пошуку у векторній базі даних та системах Retrieval-Augmented Generation (RAG)) виконати аналіз можливості застосування різних алгоритмічних стратегій:

**Divide and Conquer** (Розділяй і володарюй)

**Dynamic Programming** (Динамічне програмування)

**Greedy Strategy** (Жадібна стратегія)

**Backtracking** (Пошук з поверненням)

**Branch-and-Bound** (Метод гілок і меж)

Необхідно визначити, які алгоритмічні стратегії можуть бути використані для розв'язання поставленої задачі, проаналізувати їх переваги та недоліки, дослідити їх ефективність експериментально й обґрунтувати вибір найбільш доцільного підходу залежно від функціональних і нефункціональних вимог програмної системи.

#### 3. Аналіз задачі (Варіант 18: Оптимізація пошуку у векторній базі даних та системах RAG).

##### 3.1. Змістовна та математична постановка задачі
У сучасних системах Retrieval-Augmented Generation (RAG) текстові документи розбиваються на фрагменти (*chunks*), кожен із яких перетворюється нейромережевою моделлю-енкодером у багатовимірний числовий вектор (*embedding*) $v_i \in \mathbb{R}^d$. Під час надходження запиту користувача система перетворює запит у вектор $q \in \mathbb{R}^d$ і розв'язує двофазну задачу оптимізації пошуку та відбору контексту:

1. Фаза І — Пошук найближчих сусідів у векторній базі даних (Top-$k$ / $M$ Vector Search)

   Серед великої множини векторів $V = \{v_1, v_2, \dots, v_N\} \subset \mathbb{R}^d$ необхідно знайти $M$ векторів, які максимізують міру семантичної близькості $\text{sim}(q, v_i)$ (або мінімізують метричну відстань $\text{dist}(q, v_i) = \|q - v_i\|_2$). При лінійному переборі (*Brute-Force*) складність становить $O(N \cdot d)$, що є неприйнятним для мільйонів документів у реальному часі.

2. Фаза ІІ — Оптимізація наповнення контекстного вікна LLM (Budgeted RAG Context Selection)

   Кожен із $M$ знайдених фрагментів-кандидатів має оцінку релевантності $r_i = \text{sim}(q, v_i) > 0$ та вартість у токенах $w_i \in \mathbb{N}$. Оскільки контекстне вікно великої мовної моделі (LLM) обмежене бюджетом $B$ токенів (а вартість та затримка інференсу LLM зростають зі збільшенням кількості токенів), необхідно обрати таку підмножину фрагментів $S \subseteq \{1, \dots, M\}$, яка максимізує сумарну інформаційну релевантність (з урахуванням штрафу за семантичне дублювання між фрагментами за принципом *Maximal Marginal Relevance, MMR*) без перевищення ліміту $B$.

##### 3.2. Вхідні та вихідні дані
- Вхідні дані:
  - $V = \{v_1, v_2, \dots, v_N\}$, де $v_i \in \mathbb{R}^d$ — база векторних ембедінгів документів ($N$ — кількість векторів, $d$ — розмірність простору, типово $d \in [384, 3072]$);
  - $W = \{w_1, w_2, \dots, w_N\}$, $w_i \in \mathbb{N}$ — довжина кожного текстового фрагмента у токенах;
  - $q \in \mathbb{R}^d$ — вектор ембедінгу користувацького запиту;
  - $k \in \mathbb{N}$ (або розмір пулу кандидатів $M \ge k$) — цільова кількість релевантних векторів;
  - $B \in \mathbb{N}$ — максимальний бюджет токенів контекстного вікна LLM;
  - $\lambda \in [0, 1]$ — коефіцієнт балансу між релевантністю запиту та різноманітністю контексту (у критерії MMR).
- Вихідні дані:
  - Впорядкована підмножина індексів фрагментів $S^* = (i_1, i_2, \dots, i_m) \subseteq \{1, \dots, N\}$, їхні оцінки близькості $\{r_{i}\}_{i \in S^*}$ та сформований текстовий контекст сумарною довжиною $\sum_{i \in S^*} w_i \le B$.

##### 3.3. Критерій оптимальності
- Для Фази І (векторний індексний пошук): мінімізація відстані до $k$ найближчих сусідів (або максимізація косинусної подібності) та максимізація метрики повноти відбору $\text{Recall@}k = \frac{|S_{\text{alg}} \cap S_{\text{exact}}|}{|S_{\text{exact}}|}$ за мінімального часу відповіді $T(q) \to \min$:
  $$S_k^* = \arg\max_{S \subseteq V,\, |S|=k} \sum_{v \in S} \frac{\langle q, v \rangle}{\|q\|_2 \|v\|_2}$$
- Для Фази ІІ (оптимізація контексту RAG з обмеженням токенів): максимізація цільової функції інформаційної корисності контексту:
  $$F(S) = \lambda \sum_{i \in S} r_i - (1 - \lambda) \sum_{\substack{i, j \in S \\ i < j}} \text{sim}(v_i, v_j) \longrightarrow \max_{S \subseteq C}$$
  (в окремому адитивному випадку при $\lambda = 1$: класична цільова функція $F(S) = \sum_{i \in S} r_i \to \max$ за умови $\sum_{i \in S} w_i \le B$).

##### 3.4. Основні обмеження задачі
1. Часові обмеження (Latency SLA): у продуктивних RAG-системах етап векторного пошуку та відбору контексту повинен виконуватися за $1\text{–}20\text{ мс}$ (у Real-Time API — $< 5\text{ мс}$), оскільки основний бюджет часу витрачається на генерацію відповіді LLM.
2. Прокляття розмірності (Curse of Dimensionality): при $d = 768\text{–}1536$ об'єм гіперсфери концентрується поблизу її поверхні, а попарні відстані між випадковими точками вирівнюються, що різко знижує ефективність класичних просторових дерев точного пошуку.
3. Обмеження пам'яті (RAM Footprint): зберігання $N = 10^7$ векторів розмірності $d = 1024$ у форматі `float32` потребує $\approx 40\text{ ГБ}$ лише сирих даних, плюс накладні витрати на структуру індексу (граф або дерево).
4. Обмеження контекстного бюджету та якості: сумарна довжина відібраних фрагментів не може перевищувати ліміт токенів $\sum_{i \in S} w_i \le B$, а втрата точності відбору ($1 - \text{Recall@}k$) має залишатися в межах допустимої похибки, щоб уникнути галюцинацій LLM.

#### 4. Аналіз алгоритмічних стратегій.

##### 4.1. Divide and Conquer (Розділяй і володарюй)
- Можливість застосування: так (широко застосовується для просторового розбиття та шардування векторних індексів: KD-Tree, ієрархічна кластеризація K-Means Tree, інвертовані файлові індекси IVF — Inverted File Index, а також розподілений MapReduce / Scatter-Gather пошук по шардах БД).
- Принцип у RAG / Vector DB: векторний простір $\mathbb{R}^d$ рекурсивно або однорівнево розбивається на $K$ підпросторів (кластерів Вороного з центроїдами $c_1, \dots, c_K$). Запит перевіряє лише $n_{\text{probe}} \ll K$ найближчих кластерів, а результати з незалежних підмножин/шардів зливаються процедурою Top-$k$ Merge.
- Переваги: суттєво скорочує простір перебору з $N$ до $\frac{n_{\text{probe}}}{K} N$; ідеально піддається горизонтальному масштабуванню та паралельному виконанню на кластері серверів; помірні витрати пам'яті $O(N \cdot d + K \cdot d)$.
- Недоліки: крайовий ефект (boundary problem) - коли вектор запиту $q$ лежить поблизу межі комірки Вороного, справжній найближчий сусід може опинитися в сусідньому кластері, який не увійшов до $n_{\text{probe}}$; у просторах високої розмірності ($d > 100$) класичні KD-дерева деградують.
- Можливість отримання оптимального розв'язку: отримує наближений розв'язок при $n_{\text{probe}} < K$ у кластерних індексах IVF (або оптимальний при розподіленому Scatter-Gather по всіх шардах без відсікання кластерів).
- Складність реалізації: середня. Часова складність пошуку в IVF: $O(K \cdot d + \frac{n_{\text{probe}}}{K} N \cdot d)$; побудова індексу: $O(I \cdot K \cdot N \cdot d)$.

##### 4.2. Dynamic Programming (Динамічне програмування)
- Можливість застосування: частково (непридатна для прямого геометричного пошуку у неперервному просторі $\mathbb{R}^d$ на всій базі з мільйонів векторів, але ефективно застосовується на Фазі ІІ — для точного наповнення контекстного вікна RAG за схемою дискретного 0/1 рюкзака за лімітом токенів $B$, а також у задачах динамічного вирівнювання текстових послідовностей і оптимального розбиття документа на чанки).
- Обґрунтування обмеження для Фази І: метричний пошук найближчих сусідів у $\mathbb{R}^d$ не має властивості перекриття дискретних підзадач (overlapping subproblems): відстань до вектора $v_i$ не декомпозується на повторно використовувані підзадачі між різними векторами бази без попередньої векторної квантизації.
- Переваги (для Фази ІІ — відбору контексту): гарантує $100\%$ математично оптимальний вибір підмножини чанків при адитивній функції релевантності ($\lambda = 1$) у межах бюджету $B$ токенів за псевдополіноміальний час.
- Недоліки: таблиця станів $DP[M][B]$ вимагає $O(M \cdot B)$ часу та пам'яті (або $O(B)$ пам'яті без відновлення шляху); при врахуванні попарної схожості чанків (нелінійний критерій MMR при $\lambda < 1$) втрачається принцип оптимальності Беллмана (оптимальність підструктури залежить від усього вже обраного набору), що призводить до експоненційного вибуху станів $O(2^M)$.
- Можливість отримання оптимального розв'язку: оптимальний (лише для адитивної задачі пакування чанків у вікно з бюджетом $B$).
- Складність реалізації: середня. Часова складність: $O(M \cdot B)$, просторова: $O(M \cdot B)$.

##### 4.3. Greedy Strategy (Жадібна стратегія)
- Можливість застосування: так (основна промислова стратегія) — лежить в основі найефективніших сучасних графових індексів векторних БД (NSW / HNSW — Hierarchical Navigable Small World, DiskANN/Vamana), векторної квантизації та алгоритму MMR (Maximal Marginal Relevance) для формування різноманітного контексту RAG.
- Принцип у RAG / Vector DB:
  1. У графовому індексі HNSW/NSW кожен вектор є вершиною графа, з'єднаною з найближчими сусідами. Пошук починається зі вхідної вершини і на кожному кроці жадібно переходить до сусіда, який мінімізує відстань до запиту $q$ (best-first / beam search з шириною променя `efSearch`).
  2. На етапі формування контексту RAG алгоритм жадібно додає фрагмент із максимальною граничною корисністю (або відношенням $\text{MMR}_i / w_i$), доки не вичерпано ліміт токенів $B$.
- Переваги: гадвисока швидкодія — логарифмічне масштабування часу пошуку $O(d \cdot \log N)$; мінімальна затримка ($< 1\text{–}5\text{ мс}$ навіть на мільйонах векторів); природно підтримує нелінійний критерій різноманітності MMR за час $O(M \cdot k)$.
- Недоліки: можливість застрягання в локальному мінімумі графа (частково нівелюється ієрархічними довгими ребрами в HNSW та параметром `efSearch`); сам граф HNSW потребує додаткової оперативної пам'яті $O(N \cdot M_{\text{deg}})$ для зберігання списків суміжності.
- Можливість отримання оптимального розв'язку: наближений (високоточний апроксимаційний розв'язок із $\text{Recall@}k = 0.95\text{–}0.99+$).
- Складність реалізації: від простої (жадібний MMR, базовий NSW) до помірної/високої (повноцінний багатошаровий HNSW). Часова складність запиту: $O(d \cdot \log N)$.

##### 4.4. Backtracking (Пошук з поверненням)
- Можливість застосування: ні (неефективна для промислових векторних БД; обмежено застосовна лише на мікрообсягах даних $M \le 20$ для повного перебору допустимих комбінацій контексту зі складними логічними обмеженнями метаданих).
- Обґрунтування обмеження: Backtracking виконує обхід дерева рішень у глибину (DFS) з відсіканням лише недопустимих станів (наприклад, коли сума токенів перевищила бюджет $\sum w_i > B$), але не використовує оцінок верхньої/нижньої межі цільової функції. Через це для знаходження найкращого розв'язку алгоритм змушений перевірити майже всі допустимі підмножини або шляхи, що дає експоненційну складність $O(2^M)$ або $O(N)$ без прискорення.
- Переваги: мінімальні витрати додаткової пам'яті на стек рекурсії $O(M)$; простота реалізації; можливість знайти всі допустимі комбінації фрагментів, що задовольняють жорсткі правила фільтрації.
- Недоліки: катастрофічне експоненційне зростання часу виконання $O(2^M)$; повна непридатність для великих баз ембедінгів та систем реального часу.
- Можливість отримання оптимального розв'язку: оптимальний (або множина всіх допустимих розв'язків), але досяжний лише при дуже малих розмірах задачі.
- Складність реалізації: низька (проста рекурсивна схема DFS). Часова складність: $O(2^M)$.

##### 4.5. Branch-and-Bound (Метод гілок і меж)
- Можливість застосування: частково (застосовується для точного пошуку найближчих сусідів у метричних деревах — Ball-Tree, VP-Tree, KD-Tree з відсіканням гілок, а також для точного комбінаторного відбору контексту RAG).
- Принцип у RAG / Vector DB: простір векторів організується у дерево вкладених гіперсфер (Ball-Tree), де кожен вузол має центр $c$ і радіус $R$. За нерівністю трикутника будь-який вектор $v$ у піддереві задовольняє $\|q - v\|_2 \ge \|q - c\|_2 - R$. Якщо ця нижня межа більша за поточну відстань до $k$-го знайденого сусіда $d_{\max}$, усе піддерево гарантовано відсікається без втрати точності! Аналогічно при пакуванні контексту відсікаються гілки, де релаксаційна верхня межа менша за рекорд.
- Обґрунтування обмеження у високих розмірностях: через прокляття розмірності при $d = 512\text{–}1536$ радіуси піддерев $R$ стають сумірними з відстанню $\|q - c\|_2$, нижня межа $\|q - c\|_2 - R$ стає близькою до нуля, відсікання майже перестає спрацьовувати, і час пошуку деградує в найгіршому випадку до повного перебору $O(N \cdot d)$.
- Переваги: гарантує $100\%$ точний оптимальний розв'язок ($\text{Recall@}k = 1.0$); дуже швидкий при малій розмірності простору ($d \le 20\text{–}32$) або сильних оцінках меж.
- Недоліки: різка втрата швидкодії при високій розмірності ембедінгів ($d \ge 128$); непередбачуваний (варіативний) час відгуку залежно від розташування запиту.
- Можливість отримання оптимального розв'язку: оптимальний ($100\%$ точність).
- Складність реалізації: висока (побудова збалансованих метричних дерев, черги з пріоритетами, обчислення релаксаційних оцінок). Часова складність: у середньому $O(d \cdot N^{1 - 1/d})$ (до $O(d \log N)$ при малих $d$), у найгіршому випадку — $O(N \cdot d)$ або $O(2^M)$.

---

##### **Таблиця теоретичного аналізу алгоритмічних стратегій**

| Алгоритмічна стратегія | Можливість застосування (Так / Частково / Ні) | Обґрунтування | Характер розв'язку (Оптимальний / Допустимий / Наближений) | Переваги | Недоліки |
| :--- | :---: | :--- | :---: | :--- | :--- |
| **Divide and Conquer** | **Так** | Розбиття векторного простору на кластери/підпростори (IVF, ієрархічний K-Means) та шардування БД із подальшим злиттям Top-$k$. | Наближений (IVF) / Оптимальний (повний Scatter-Gather) | Висока масштабованість, природна паралелізація між вузлами, суттєве зменшення кількості порівнянь. | Крайові ефекти на межах кластерів (зниження Recall), деградація координатних розбиттів при великих $d$. |
| **Dynamic Programming** | **Частково** | Непридатна для метричного пошуку в $\mathbb{R}^d$ (немає перекриття підзадач), але розв'язує Фазу ІІ — точне пакування чанків у ліміт $B$ токенів (0/1 Knapsack). | Оптимальний (для адитивного бюджету токенів) | Гарантує точний максимум сумарної релевантності в межах вікна $B$ за час $O(M \cdot B)$. | Великі витрати пам'яті $O(M \cdot B)$; не працює поліноміально при нелінійному штрафі за дублікати (MMR). |
| **Greedy Strategy** | **Так** | Основа графової навігації HNSW/NSW у векторних БД та алгоритму MMR для швидкого формування різноманітного контексту RAG. | Наближений | Сублінійний час пошуку $O(d \log N)$, затримка $< 5\text{ мс}$, висока точність ($\text{Recall@}k > 0.95$), відмінна масштабованість. | Відсутність гарантії глобального оптимуму (ризик локальних мінімумів), додаткова пам'ять на ребра графа HNSW. |
| **Backtracking** | **Ні** | Відсікає лише порушення бюджету без оцінки перспективності гілок; експоненційний перебір непридатний для векторних БД. | Оптимальний / Допустимий (усі допустимі набори) | Простота реалізації, мінімум пам'яті $O(M)$, здатність перелічити всі допустимі комбінації при малих $M$. | Експоненційний час $O(2^M)$, повна непридатність для великих баз даних і реального часу. |
| **Branch-and-Bound** | **Частково** | Точний пошук у метричних деревах (Ball-Tree, VP-Tree) за нерівністю трикутника та точний комбінаторний відбір контексту з верхніми межами. | Оптимальний | Гарантована $100\%$ точність ($\text{Recall@}k = 1.0$), ефективне відсікання гілок у просторах малої розмірності ($d \le 32$). | Прокляття розмірності: при $d = 768\text{–}1536$ межі слабшають і складність деградує до $O(N \cdot d)$; висока складність реалізації. |

#### 5. Вибір алгоритмічної стратегії.

На основі проведеного теоретичного аналізу найбільш доцільною базовою алгоритмічною стратегією для оптимізації пошуку у векторній базі даних та системах RAG обрано Greedy Strategy (Жадібну стратегію) (зокрема, жадібний графовий пошук із променем HNSW / NSW для Фази І та жадібний відбір MMR для Фази ІІ), яку в розподілених сховищах надвеликих масштабів доцільно доповнювати стратегією Divide and Conquer (кластерне розбиття IVF та шардування даних).

##### Обґрунтування вибору за ключовими критеріями:
1. Вимоги до точності результату: графова жадібна навігація (HNSW) із параметром ширини пошуку `efSearch` забезпечує $\text{Recall@}10 = 0.95\text{–}0.995$, а жадібний MMR досягає $>95\%$ від глобального максимуму цільової функції контексту.
2. Необхідність отримання оптимального розв'язку: у системах RAG самі векторні ембедінги є неперервною статистичною апроксимацією семантики тексту. Тому математично точний пошук ($100\%$ Recall за рахунок повільного перебору) не дає статистично значущого приросту якості генерації LLM порівняно з наближеним пошуком (ANN) із $\text{Recall@}k \ge 0.96$. Строгий глобальний оптимум не є обов'язковим у стандартному конвеєрі RAG.
3. Часові обмеження: RAG-системи працюють в інтерактивному режимі, де на векторний пошук виділяється лише $1\text{–}15\text{ мс}$. Greedy-пошук у графі HNSW має сублінійну часову складність $O(d \cdot \log N)$, що на порядки швидше за лінійний скан або деградований у високих розмірностях Branch-and-Bound.
4. Обмеження пам'яті: графовий індекс зберігає самі вектори $O(N \cdot d)$ та списки суміжності фіксованого ступеня $M_{\text{deg}}$ ($O(N \cdot M_{\text{deg}})$). За дефіциту RAM жадібний пошук легко поєднується зі стисненням векторів (Product Quantization / Scalar Quantization), зменшуючи споживання пам'яті у $4\text{–}16$ разів.
5. Масштабованість рішення: логарифмічне зростання кількості обчислень відстані при збільшенні $N$ з $10^4$ до $10^8$ векторів забезпечує плавне масштабування без різкого падіння пропускної здатності (QPS).
6. Придатність до обробки великих обсягів даних: на відміну від Backtracking ($O(2^M)$) чи Dynamic Programming ($O(M \cdot B)$, непридатного для первинної фільтрації мільйонів векторів), Greedy Strategy у поєднанні з кластерним розбиттям обробляє мільярди векторів (системи рівня Milvus, Qdrant, Pinecone, FAISS).
7. Складність програмної реалізації: жадібний відбір контексту MMR та однорівневий графовий пошук NSW мають помірну й добре структуровану програмну реалізацію, а для багатошарового HNSW існують стандартизовані бібліотеки (`faiss`, `hnswlib`).
8. Можливість паралельного виконання: обробка незалежних користувацьких запитів паралелізується без блокувань по потоках CPU/GPU (read-only граф під час пошуку), обчислення скалярних добутків векторизується інструкціями SIMD (AVX-512 / NEON), а при шардуванні (Divide and Conquer) запити виконуються паралельно на багатьох вузлах.

##### **Матриця оцінювання стратегій (за шкалою 1–5)**
*(1 — найгірший показник, 5 — найкращий показник для задачі пошуку у векторній БД та RAG)*

| Стратегія | Точність | Швидкодія | Пам'ять | Масштабованість | Сумарна оцінка |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Divide and Conquer** | 4 | 4 | 4 | 5 | **17** |
| **Dynamic Programming** | 5 | 3 | 2 | 2 | **12** |
| **Greedy Strategy** | 4 | 5 | 4 | 5 | **18** |
| **Backtracking** | 5 | 1 | 4 | 1 | **11** |
| **Branch-and-Bound** | 5 | 2 | 3 | 2 | **12** |

> Підсумок за матрицею: найвищу сумарну оцінку (18 із 20 балів) отримує Greedy Strategy завдяки максимальній швидкодії (`5`) та масштабованості (`5`) при високій точності (`4`) й помірних витратах пам'яті (`4`). Друге місце (17 балів) посідає Divide and Conquer, що пояснює, чому в індустріальних векторних СУБД найчастіше використовують саме графові жадібні алгоритми (HNSW) або гібрид Divide-and-Conquer + Greedy (IVF-HNSW / IVF-PQ).

#### 6. Аналіз придатності для різних типів програмних систем.

Обрана Greedy Strategy (зокрема HNSW + жадібний MMR) демонструє високу універсальність у продуктивних системах штучного інтелекту, проте залежно від архітектурних обмежень конкретної програмної системи (жорсткий ліміт RAM на периферійних пристроях, розподілені петабайтні кластери чи вимога строго доведеного глобального оптимуму) доцільно адаптувати стратегію або комбінувати її з іншими підходами.

##### **Таблиця придатності та рекомендованих стратегій для різних типів програмних систем**

| Тип програмної системи | Рекомендована алгоритмічна стратегія | Обґрунтування вибору |
| :--- | :--- | :--- |
| Вбудована система (Edge AI / локальний RAG на пристрої) | Divide and Conquer (компактний IVF / KD-розбиття з квантизацією) + Greedy Strategy (жадібний MMR без важкого графа) | Вбудовані системи мають жорсткі обмеження оперативної пам'яті, тому зберігати багатошаровий граф HNSW у RAM дорого; кластерне розбиття IVF зі стисненими векторами та простий жадібний добір чанків мінімізують споживання пам'яті при прийнятній швидкості. |
| Високонавантажена вебсистема (Search / Chat API з високим QPS) | Greedy Strategy (HNSW + жадібний MMR) | Критичними є мінімальна затримка ($< 5\text{–}10\text{ мс}$) та висока пропускна здатність при тисячах одночасних запитів. Жадібний обхід графа HNSW забезпечує логарифмічний час $O(d \log N)$ і високий $\text{Recall@}k$. |
| Big Data-застосування (Корпоративні сховища $10^8\text{–}10^9$ векторів) | Divide and Conquer (шардування + IVF) у поєднанні з Greedy Strategy | Один вузол не вміщує весь індекс у RAM. Стратегія Divide and Conquer розбиває векторний простір між вузлами кластера та кластерами Вороного, а всередині шарду/кластера застосовується швидкий жадібний відбір. |
| Хмарний сервіс (Multi-tenant SaaS RAG / Vector DBaaS) | Greedy Strategy + Divide and Conquer (ізольовані партиції / HNSW на шард) | Хмарні сервіси вимагають еластичного горизонтального масштабування між орендарями (Divide and Conquer за namespace/shard) та гарантованого низького часу відгуку SLA (Greedy HNSW). |
| Система підтримки прийняття рішень (Аналітичний / юридичний / медичний RAG) | Greedy Strategy (широкий пошук HNSW) + Dynamic Programming (точне пакування вікна контексту) | Потребує високої повноти та збалансованого наповнення контекстного вікна аналітичними фактами: HNSW з підвищеним `efSearch` знаходить пул кандидатів, а DP точно максимізує інформаційну цінність у межах бюджету токенів $B$. |
| Система реального часу (Голосові AI-асистенти, стрімінговий RAG) | Greedy Strategy (NSW / HNSW з жорстким лімітом кроків) | Має детермінований верхній поріг часу відповіді (жорсткий бюджет $< 5\text{ мс}$ на retrieval). Жадібний пошук дозволяє зафіксувати максимальну кількість обчислень відстані та миттєво повернути найкращий знайдений контекст. |
| Оптимізаційна система (Точний аудит даних, еталонна верифікація індексів) | Branch-and-Bound (метричні дерева / релаксаційні межі) або Dynamic Programming | Коли головним критерієм є математично гарантована $100\%$ оптимальність розв'язку (нульова похибка відносно еталона), а час обчислення менш критичний, використовують точні методи відсікання гілок або динамічне програмування. |

#### 7. Формування рекомендацій щодо вибору алгоритмічної стратегії.

##### Експеримент 1: Визначення області ефективного застосування алгоритмічних стратегій

1. Малі розміри задач ($N \le 1\,000$ векторів або $M \le 15$ фрагментів контексту):
   на малих пулах чанків ($M \le 15$) виправдане використання точних методів — Branch-and-Bound ($0.06\text{–}0.08\text{ мс}$), Dynamic Programming($2\text{–}5\text{ мс}$ при невеликому $B$) та навіть Backtracking (якщо потрібно знайти всі допустимі підмножини чанків під складні бізнес-фільтри). Для векторної бази до $1\,000$ елементів витрати на обхід графа співмірні з прямим векторизованим перебором.
2. Великі розміри задач ($N \ge 10^5\text{–}10^9$ векторів, великі пули кандидатів):
   ефективними є виключно Greedy Strategy (навігація в графі HNSW за $O(d \log N)$ + жадібний MMR) та Divide and Conquer (кластеризація IVF, шардування індексу по вузлах). Точні експоненційні або табличні методи тут повністю непридатні.
3. Жорсткі обмеження пам'яті:
   на етапі векторного пошуку найкраще підходить Divide and Conquer (IVF без зберігання важкого графа суміжності + стиснення векторів); на етапі точного комбінаторного добору з малого пулу — Branch-and-Bound або Greedy Strategy (потребують лише $O(M)$ пам'яті — $2\text{–}8\text{ КБ}$ проти сотень КБ / десятків МБ у таблиці DP).
4. Необхідність отримання оптимального рішення ($100\%$ точність):
   для просторів малої розмірності ($d \le 32$) або великих бюджетів токенів $B$ — Branch-and-Bound (точне відсікання за нерівністю трикутника в Ball-Tree або дробовою релаксацією); для адитивного пакування контексту при помірному дискретному бюджеті $B$ та корельованих вагах — Dynamic Programming.
5. Необхідність швидкого наближеного рішення (Real-Time RAG):
   безальтернативним лідером є Greedy Strategy: графовий пошук HNSW/NSW виконується за $\sim 1\text{ мс}$, а жадібне наповнення вікна контексту — за $\sim 0.004\text{ мс}$ при середній точності $98\text{–}99\%$ від глобального максимуму.

##### **Таблиця рекомендацій щодо вибору стратегії за вимогами програмної системи**

| Вимога програмної системи | Рекомендована стратегія | Обґрунтування |
| :--- | :--- | :--- |
| Мінімальний час виконання | Greedy Strategy | Логарифмічна навігація у графі HNSW $O(d \log N)$ та жадібний відбір чанків за $O(M \log M)$ забезпечують субмілісекундний/мілісекундний відгук. |
| Мінімальне використання пам'яті | Greedy Strategy (без графа, на базі скану/квантизації) або Branch-and-Bound (для Фази ІІ) | Жадібне сортування та рекурсивний обхід у глибину в Branch-and-Bound потребують лише $O(M)$ додаткової пам'яті (проти $O(M \cdot B)$ у DP та додаткових ребер графа). |
| Гарантована оптимальність розв'язку | Branch-and-Bound / Dynamic Programming | Branch-and-Bound гарантує $\text{Recall@}k = 1.0$ у метричних деревах і точний оптимум пакування без втрати допустимих рішень; DP гарантує $100\%$ оптимум при адитивному бюджеті $B$. |
| Великі обсяги вхідних даних | Divide and Conquer + Greedy Strategy | Шардування та кластеризація IVF ділять мільярдні бази на незалежні партиції, а жадібний графовий пошук забезпечує швидку навігацію всередині них. |
| Комбінаторна оптимізація (вибір найкращого набору фрагментів) | Branch-and-Bound (точна) / Greedy MMR (швидка нелінійна) | Branch-and-Bound ефективно відсікає $>99\%$ неперспективних комбінацій чанків; при нелінійному критерії різноманітності MMR ефективно працює жадібна субмодульна максимізація. |
| Гарантоване знаходження розв'язку | Dynamic Programming (Фаза ІІ) / Branch-and-Bound (Фаза І та ІІ) | Завжди знаходять математично точний допустимий та оптимальний розв'язок за скінченну кількість кроків без ризику застрягти в локальному мінімумі чи пропустити кластер. |
| Швидке наближене рішення | Greedy Strategy | Забезпечує найкращий компроміс «швидкість–точність»: прискорення у десятки/сотні разів при похибці лише $1\text{–}5\%$. |
| Обмежені обчислювальні ресурси | Greedy Strategy + Divide and Conquer (IVF) | Пропускають $>95\%$ обчислень евклідової/косинусної відстані та не створюють великих таблиць станів у пам'яті. |
| Можливість паралельного виконання | Divide and Conquer | Розбиття бази на незалежні кластери або шарди дозволяє виконувати пошук паралельно на багатьох ядрах CPU, GPU чи вузлах кластера з фінальним злиттям Top-$k$. |

##### Експеримент 2: Матриця вибору алгоритмічної стратегії

| Вимога системи | Рекомендована стратегія | Обґрунтування |
| :--- | :--- | :--- |
| Сценарій 1: система реального часу (Реальний час — голосові AI-агенти, інтерактивний автокомпліт)| Greedy Strategy (HNSW / NSW з обмеженим `efSearch` + жадібний MMR) | Має передбачувано низьку затримку ($< 1.5\text{ мс}$ на пошук та $< 0.01\text{ мс}$ на відбір контексту в експерименті). Кількість кроків жадібного обходу жорстко обмежується параметром `efSearch`, що гарантує відсутність перевищення часового SLA. |
| Сценарій 2: система з жорсткими ресурсними обмеженнями (Обмежена пам'ять — мобільний/вбудований локальний RAG) | Divide and Conquer (кластерний індекс IVF зі стисненням) + Greedy Strategy (жадібний добір без графа HNSW); для точного добору з малого пулу — Branch-and-Bound | Граф HNSW вимагає значної додаткової RAM на списки суміжності, а таблиця DP — $O(M \cdot B)$ пам'яті. Натомість IVF зберігає лише $K \ll N$ центроїдів, а жадібне сортування чи DFS у Branch-and-Bound використовують лише $O(M)$ пам'яті (кілька кілобайтів). |
| Сценарій 3: високонавантажена інформаційна система (Пропускна здатність / Масштабованість / Продуктивність) | Divide and Conquer (горизонтальне шардування + IVF) + Greedy Strategy (HNSW на кожному шарді) | Шардування за принципом «розділяй і володарюй» лінійно масштабує пропускну здатність (QPS) між серверами кластера, а жадібна графова навігація мінімізує навантаження на CPU кожного вузла (перевіряється $<5\%$ векторів). |
| Сценарій 4: система, для якої критично важлива оптимальність розв'язку (Оптимальність — фінансовий комплаєнс, юридична експертиза) | Branch-and-Bound (точний пошук у метричних деревах / відсікання за верхніми межами) або Dynamic Programming (при помірному бюджеті $B$) | Виключають евристичну втрату релевантних документів ($\text{Recall@}k = 1.0$, $100\%$ оптимум цільової функції контексту), при цьому працюючи значно швидше за повний перебір у просторах малої/середньої розмірності та на етапі пакування контексту. |
| Великий простір пошуку та велика кількість альтернатив (Мільйони ембедінгів і сотні фрагментів-кандидатів) | Greedy Strategy (ієрархічний HNSW + жадібний MMR) у поєднанні з Divide and Conquer | Експоненційні методи (Backtracking, точний BnB при великих $d$) і табличний DP зазнають комбінаторного вибуху. Тільки ієрархічне звуження простору та жадібний графовий рух знижують складність до логарифмічної $O(d \log N)$. |
| Необхідність знаходження всіх допустимих розв'язків (Повний аудит усіх комбінацій документів, що задовольняють політики доступу та бюджет) | Backtracking (лише при малому пулі попередньо відфільтрованих кандидатів $M \le 20$) | Єдина стратегія, яка за своєю природою систематично обходить дерево комбінацій із відсіканням порушень обмежень ($\sum w_i > B$ або конфліктів метаданих) і здатна згенерувати повний перелік усіх допустимих наборів контексту. |

##### Експеримент 3. Аналіз ризиків використання алгоритмічної стратегії

| Алгоритмічна стратегія | Основний ризик | Можливі наслідки |
| :--- | :--- | :--- |
| Divide and Conquer | Крайовий ефект (Boundary Problem) та незбалансованість кластерів: вектор запиту потрапляє на межу комірки Вороного в індексі IVF, а найближчі сусіди опиняються в сусідньому кластері поза `n_probe`; утворення перевантажених («гарячих») кластерів. | Пропуск релевантних фрагментів документа (падіння $\text{Recall@}k$), нерівномірне навантаження на вузли кластера та стрибки затримки при зверненні до великих партицій. |
| Dynamic Programming | Псевдополіноміальне розростання таблиці станів $O(M \cdot B)$ при великих контекстних вікнах LLM ($B = 32\text{k–}128\text{k}$ токенів) та неможливість врахувати попарну схожість чанків (MMR) без експоненційного стану $O(2^M)$. | Надмірне споживання оперативної пам'яті (ризик Out-Of-Memory при паралельних запитах), суттєве уповільнення відбору контексту або дублювання однотипних фрагментів у промпті LLM. |
| Greedy Strategy | Застрягання в локальному мінімумі графа близькості (HNSW / NSW) при недостатній ширині променя `efSearch`, а на етапі пакування контексту за щільністю $r_i / w_i$ — залишок невикористаного бюджету токенів. | Втрата невеликої частки ($1\text{–}6\%$) найближчих векторів або незначний недобір сумарної релевантності контексту, що для складних багатоходових запитів може знизити повноту відповіді LLM. |
| Backtracking | Експоненційний (комбінаторний) вибух простору пошуку $O(2^M)$ через відсутність оцінок верхньої/нижньої межі цільової функції. | Критичне зростання часу відповіді вже при $M > 20\text{–}25$, зависання обробників запитів (Timeout) та відмова в обслуговуванні продуктивної RAG-системи. |
| Branch-and-Bound | Деградація оцінок меж через «прокляття розмірності» у просторах ембедінгів ($d = 768\text{–}1536$), коли нижня межа $\|q - c\|_2 - R$ прямує до нуля, або ослаблення верхньої межі при однаковій щільності чанків ($r_i / w_i \approx \text{const}$). | Відсікання гілок перестає ефективно спрацьовувати, час виконання непередбачувано вироджується до повного перебору $O(N \cdot d)$ або $O(2^M)$, порушуючи вимоги SLA. |

##### Рекомендації щодо використання обраної алгоритмічної стратегії у різних сценаріях

1. Сценарій 5 (Real-Time API / High-Load): Обмеження по часу $< 5\text{ мс}$, допустима похибка $< 10\%$.
   Рекомендована стратегія: Greedy Strategy (жадібна навігація з променем у графі близькості HNSW / NSW на Фазі І + жадібний відбір фрагментів за питомою корисністю / MMR на Фазі ІІ).
   Обґрунтування вибору:
     - Дотримання обмеження по часу ($< 5\text{ мс}$): на великих корпусах документів лінійний перебір $O(N \cdot d)$ та точні методи (Branch-and-Bound у високих розмірностях $d \ge 512$, Dynamic Programming при великому бюджеті $B$) потребують десятків або сотень мілісекунд. Жадібний обхід графа HNSW забезпечує логарифмічну складність $O(d \cdot \log N)$, перевіряючи лише малу частку вершин графа і виконуючи пошук за $1\text{–}3\text{ мс}$, а жадібне формування контексту за $O(M \log M)$ або $O(M \cdot k)$ займає частки мілісекунди.
     - Прийнятність похибки ($< 10\%$): графовий жадібний пошук із помірним параметром `efSearch` стабільно забезпечує повноту $\text{Recall@}k = 0.94\text{–}0.99$ (похибка $1\text{–}6\% < 10\%$), а жадібний добір фрагментів у контекстне вікно дає відхилення від глобального оптимуму в межах $1\text{–}4\%$. Оскільки векторні ембедінги самі є статистичною апроксимацією семантики, похибка до $5\text{–}10\%$ не погіршує якість фінальної генерації LLM, зате дозволяє витримувати тисячі запитів за секунду (High-Load).

2. Сценарій 6 (Mission-Critical / Financial System): Вимога $100\%$ точності, час виконання не критичний.
   Порівняння та обґрунтування вибору між Dynamic Programming та Branch-and-Bound:
     - Вимоги до оптимальності та застосовність до фаз задачі: обидві стратегії гарантують $100\%$ математично точний оптимум там, де вони застосовні. Проте для Фази І (метричний пошук найближчих векторів у неперервному просторі $\mathbb{R}^d$) Dynamic Programming не може бути застосований через відсутність дискретних перекривних підзадач, тому для точного векторного пошуку обирається виключно Branch-and-Bound (метричні дерева Ball-Tree / VP-Tree з точним відсіканням за нерівністю трикутника). На Фазі ІІ (точне пакування контекстного вікна за адитивною релевантністю $\sum r_i \to \max$ при $\sum w_i \le B$) застосовні обидва підходи.
     - Доступні ресурси (пам'ять): Dynamic Programming потребує виділення таблиці станів обсягом $O(M \cdot B)$ для відновлення оптимальної підмножини чанків, що при великому бюджеті токенів $B$ споживає десятки мегабайтів RAM на один запит. Branch-and-Bound при пошуку в глибину потребує лише $O(M)$ оперативної пам'яті (кілька кілобайтів на стек рекурсії), тому є значно економнішим за пам'яттю.
     - Розмір задачі та масштаб бюджету $B$: складність Dynamic Programming $O(M \cdot B)$ лінійно залежить від величини бюджету $B$ — для сучасних фінансових LLM із вікном $B = 32\,000\text{–}128\,000$ токенів (або при нецілочисельних вагах) таблиця станів стає надто великою, тоді як швидкодія Branch-and-Bound не залежить від масштабу числа $B$. Водночас зі збільшенням кількості кандидатів ($M > 40\text{–}60$) у Branch-and-Bound зростає глибина дерева комбінацій.
     - Передбачуваність часу виконання: Dynamic Programming має строго детермінований час виконання $O(M \cdot B)$ для будь-яких вхідних даних (навіть коли всі фрагменти мають майже однакову щільність $r_i / w_i \approx \text{const}$). Натомість час роботи Branch-and-Bound сильно залежить від якості оцінок меж: на некорельованих даних він працює в рази швидше за DP завдяки відсіканню $>99\%$ гілок, але на «складних» сильно корельованих фінансових документах релаксаційна межа слабшає і час перебору може різко зрости.
   - Висновок: на етапі точного пошуку у векторному просторі обирається Branch-and-Bound; на етапі формування контексту RAG при помірному дискретному бюджеті токенів ($B \le 4\,000\text{–}8\,000$) та достатньому обсязі RAM доцільніше обрати Dynamic Programming через гарантовано передбачуваний час роботи на будь-яких даних, а при великих контекстних вікнах ($B \ge 32\,000$), дробових оцінках або обмеженій пам'яті — Branch-and-Bound (з ініціалізацією початкового рекорду жадібним розв'язком).

#### 8. Висновки.

У ході виконання лабораторної роботи було проведено аналіз можливості та ефективності застосування п'яти основних алгоритмічних стратегій (Divide and Conquer, Dynamic Programming, Greedy Strategy, Backtracking, Branch-and-Bound) до задачі оптимізації пошуку у векторній базі даних та системах Retrieval-Augmented Generation (RAG) (Варіант 18).

Загальні рекомендації щодо вибору основної алгоритмічної стратегії:
найбільш доцільною базовою стратегією для продуктивних систем векторного пошуку та RAG є Greedy Strategy (18 із 20 балів за матрицею оцінювання). Вона лежить в основі графових індексів наближеного пошуку найближчих сусідів (NSW / HNSW), забезпечуючи логарифмічну швидкодію $O(d \cdot \log N)$, затримку в межах кількох мілісекунд ($< 5\text{ мс}$) та високу точність відбору ($\text{Recall@}k = 0.95\text{–}0.99+$). На етапі наповнення контекстного вікна LLM жадібний підхід (зокрема за критерієм Maximal Marginal Relevance, MMR) дозволяє за частки мілісекунди відібрати релевантні та різноманітні фрагменти в межах бюджету токенів $B$ із похибкою, що зазвичай не перевищує $1\text{–}5\%$.

Умови доцільного використання альтернативних стратегій:
   - Divide and Conquer (17 із 20 балів) доцільно застосовувати у великомасштабних Big Data та хмарних системах ($N \ge 10^7\text{–}10^9$ векторів), а також у вбудованих системах із жорсткими обмеженнями оперативної пам'яті. Кластерне розбиття простору (індекси IVF) та горизонтальне шардування дозволяють зменшити витрати RAM на графові зв'язки та ефективно розпаралелити пошук між вузлами кластера (найчастіше у гібридній зв'язці Divide and Conquer + Greedy Strategy).
   - Branch-and-Bound (12 із 20 балів) є доцільним у критично важливих (Mission-Critical) та оптимізаційних системах із вимогою $100\%$ точності розв'язку: для точного метричного пошуку за помірного числа вимірів ($d \le 32$, дерева Ball-Tree / VP-Tree), а також для точного комбінаторного відбору контексту за великих бюджетів токенів ($B \ge 32\,000$) та обмеженої оперативної пам'яті ($O(M)$).
   - Dynamic Programming (12 із 20 балів) не підходить для геометричного пошуку у просторі $\mathbb{R}^d$, проте є оптимальним вибором на другій фазі конвеєра RAG у Mission-Critical системах, коли необхідно гарантовано знайти $100\%$ точний максимум адитивної релевантності чанків у межах помірного дискретного бюджету токенів ($B \le 4\,000\text{–}8\,000$) за детермінований час $O(M \cdot B)$, незалежно від корельованості вхідних даних.
   - Backtracking (11 із 20 балів) через експоненційну часову складність $O(2^M)$ є непридатним для промислових векторних БД і може використовуватися виключно на надмалих попередньо відфільтрованих вибірках ($M \le 15\text{–}20$), коли бізнес-вимога вимагає знаходження та аудиту абсолютно всіх допустимих комбінацій фрагментів документів.

Отже, вибір алгоритмічної стратегії визначається балансом між вимогами до затримки, точності та доступних ресурсів: для інтерактивних і високонавантажених RAG-систем стандартом є Greedy Strategy у поєднанні з просторовою декомпозицією Divide and Conquer, тоді як точні методи (Branch-and-Bound і Dynamic Programming) доцільно залучати точково на фінальних етапах відбору контексту в системах підвищеної відповідальності.